In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import json
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

import warnings
warnings.filterwarnings('ignore')


In [3]:
# (CHANGE THESE ACCORDING TO YOUR DRIVE)
STUDENTLIFE_PATH = "//content/drive/MyDrive/Ai_Burnout_predictions/Data"
#OUTPUT_PATH = "/content/drive/MyDrive/AI_Burnout_Predictor/results_realistic_studentlife"

#os.makedirs(OUTPUT_PATH, exist_ok=True)

print("Paths configured.")


Paths configured.


In [4]:
# Loading StudentLife data
def load_studentlife_json(folder_path):
    all_data = []
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".json"):
            student_id = file_name.replace(".json", "")
            with open(os.path.join(folder_path, file_name), "r") as f:
                records = json.load(f)
                for r in records:
                    r["student_id"] = student_id
                    all_data.append(r)
    return pd.DataFrame(all_data)

print("Loading StudentLife...")

stress_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Stress"))
activity_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Activity"))
sleep_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Sleep"))

print(f"Stress: {stress_raw.shape}")
print(f"Activity: {activity_raw.shape}")
print(f"Sleep: {sleep_raw.shape}")


Loading StudentLife...
Stress: (2408, 5)
Activity: (833, 9)
Sleep: (1644, 7)


In [5]:
stress_raw.head()

,null,resp_time,student_id,level,location
0,"43.70643559,-72.28705828",1364118627,Stress_u39,NaN,NaN
1,1,1364121576,Stress_u39,NaN,NaN
2,2,1364121579,Stress_u39,NaN,NaN
3,2,1364121823,Stress_u39,NaN,NaN
4,3,1364121578,Stress_u39,NaN,NaN


In [6]:
activity_raw.head()

,Social2,null,resp_time,student_id,other_relaxing,other_working,relaxing,working,location
0,1,1,1365393298,Activity_u56,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,1365396239,Activity_u56,4,1,1,1,NaN
2,NaN,NaN,1365496964,Activity_u56,4,1,1,1,NaN
3,NaN,NaN,1365619688,Activity_u56,2,2,1,3,NaN
4,NaN,NaN,1365660277,Activity_u56,2,2,1,3,NaN


In [7]:
sleep_raw.head()

,null,resp_time,student_id,hour,location,rate,social
0,"43.75908069,-72.32885314",1364114760,Sleep_u00,NaN,NaN,NaN,NaN
1,8,1364114765,Sleep_u00,NaN,NaN,NaN,NaN
2,1,1364114453,Sleep_u00,NaN,NaN,NaN,NaN
3,1,1364114775,Sleep_u00,NaN,NaN,NaN,NaN
4,3,1364114365,Sleep_u00,NaN,NaN,NaN,NaN


In [8]:
# Cleaning the STRESS DATASET
# =========================
print("\n DATASET: STRESS")
print("Shows self-reported student stress levels over time")

print("\nCleaning StudentLife Stress...")

stress_clean = stress_raw.copy()

# Dropping the 'null' column
if 'null' in stress_clean.columns:
    stress_clean = stress_clean.drop(columns=['null'])

print("\nInitial Stress Dataset:")
print(stress_clean)

# Converting  timestamp
stress_clean['timestamp'] = pd.to_datetime(stress_clean['resp_time'], unit='s')
print("\nAfter converting resp_time to timestamp:")
print(stress_clean)

# Cleaning  student_id
stress_clean['student_id'] = stress_clean['student_id'].str.replace('Stress_', '')
print("\nAfter cleaning student_id:")
print(stress_clean)



 DATASET: STRESS
Shows self-reported student stress levels over time

Cleaning StudentLife Stress...

Initial Stress Dataset:
       resp_time  student_id level                  location
0     1364118627  Stress_u39   NaN                       NaN
1     1364121576  Stress_u39   NaN                       NaN
2     1364121579  Stress_u39   NaN                       NaN
3     1364121823  Stress_u39   NaN                       NaN
4     1364121578  Stress_u39   NaN                       NaN
...          ...         ...   ...                       ...
2403  1369166429  Stress_u58     1  43.70638908,-72.28324332
2404  1369210324  Stress_u58     1   43.7062243,-72.28296326
2405  1369299604  Stress_u58     2  43.70694322,-72.28640413
2406  1369779460  Stress_u58     1  43.70622433,-72.28334234
2407  1369339101  Stress_u58     2  43.70605751,-72.28339763

[2408 rows x 4 columns]

After converting resp_time to timestamp:
       resp_time  student_id level                  location  \
0     1364

In [9]:
#renaming the level column
stress_clean = stress_clean.rename(columns={'level': 'stress_level'})
print("\nAfter renaming level → stress_level:")
print(stress_clean)

#Converting stress_level to numeric data
stress_clean['stress_level'] = pd.to_numeric(stress_clean['stress_level'], errors='coerce')
print("\nAfter converting stress_level to numeric:")
print(stress_clean)

if 'null' in stress_clean.columns:

    # if stress_level is missing but null looks like numeric, then use this
    null_as_num = pd.to_numeric(stress_clean['null'], errors='coerce')
    fill_mask = stress_clean['stress_level'].isna() & null_as_num.notna()
    if fill_mask.any():
        stress_clean.loc[fill_mask, 'stress_level'] = null_as_num.loc[fill_mask]

    #If location is missing but null looks like "lat,long", then use this
    if 'location' in stress_clean.columns:
        null_as_str = stress_clean['null'].astype(str)
        coord_mask = stress_clean['location'].isna() & null_as_str.str.match(
            r'^-?\d+(\.\d+)?,-?\d+(\.\d+)?$'
        )
        if coord_mask.any():
            stress_clean.loc[coord_mask, 'location'] = stress_clean.loc[coord_mask, 'null']

print("\nAfter recovering values from 'null' (if applicable):")
print(stress_clean)

#dropping missing stress values
stress_clean = stress_clean.dropna(subset=['stress_level'])
print("\nAfter dropping NaN stress levels:")
print(stress_clean)

#Explicit float conversion
stress_clean['stress_level'] = stress_clean['stress_level'].astype(float)
print("\nFinal cleaned Stress dataset:")
print(stress_clean)



After renaming level → stress_level:
       resp_time student_id stress_level                  location  \
0     1364118627        u39          NaN                       NaN   
1     1364121576        u39          NaN                       NaN   
2     1364121579        u39          NaN                       NaN   
3     1364121823        u39          NaN                       NaN   
4     1364121578        u39          NaN                       NaN   
...          ...        ...          ...                       ...   
2403  1369166429        u58            1  43.70638908,-72.28324332   
2404  1369210324        u58            1   43.7062243,-72.28296326   
2405  1369299604        u58            2  43.70694322,-72.28640413   
2406  1369779460        u58            1  43.70622433,-72.28334234   
2407  1369339101        u58            2  43.70605751,-72.28339763   

               timestamp  
0    2013-03-24 09:50:27  
1    2013-03-24 10:39:36  
2    2013-03-24 10:39:39  
3    2013-03-

In [10]:
# Code for Activity data cleaning
print("\n DATASET: ACTIVITY")
print(" Shows students' daily activity levels (social, working, relaxing)")

print("\nCleaning StudentLife Activity")

activity_clean = activity_raw.copy()

# Drop the 'null' coloumns
if 'null' in activity_clean.columns:
    activity_clean = activity_clean.drop(columns=['null'])

print("\nInitial Activity Dataset:")
print(activity_clean)

# Convert timestamp

activity_clean['timestamp'] = pd.to_datetime(activity_clean['resp_time'], unit='s')
print("\nAfter converting resp_time to timestamp:")
print(activity_clean)

# Clean student_id
activity_clean['student_id'] = activity_clean['student_id'].str.replace('Activity_', '')
print("\nAfter cleaning student_id:")
print(activity_clean)



 DATASET: ACTIVITY
 Shows students' daily activity levels (social, working, relaxing)

Cleaning StudentLife Activity

Initial Activity Dataset:
    Social2   resp_time    student_id other_relaxing other_working relaxing  \
0         1  1365393298  Activity_u56            NaN           NaN      NaN   
1       NaN  1365396239  Activity_u56              4             1        1   
2       NaN  1365496964  Activity_u56              4             1        1   
3       NaN  1365619688  Activity_u56              2             2        1   
4       NaN  1365660277  Activity_u56              2             2        1   
..      ...         ...           ...            ...           ...      ...   
828     NaN  1365746529  Activity_u02              2             3        1   
829     NaN  1366005686  Activity_u02              1             3        1   
830     NaN  1365920939  Activity_u02              3             2        1   
831     NaN  1367083019  Activity_u02              2             

In [11]:


# key columns of studentlife dataset

for col in ["Social2", "working", "other_working", "relaxing", "other_relaxing"]:
    if col not in activity_clean.columns:
        activity_clean[col] = np.nan



# Converting to numeric

for col in ["Social2", "working", "other_working", "relaxing", "other_relaxing"]:
  activity_clean[col] = pd.to_numeric(activity_clean[col], errors="coerce")



# Computing interpretable scores

activity_clean["workload_score"] = activity_clean[["working", "other_working"]].sum(axis=1, min_count=1)
activity_clean["recovery_score"] = activity_clean[["relaxing", "other_relaxing"]].sum(axis=1, min_count=1)
activity_clean["social_score"] = activity_clean["Social2"]


print("\nAfter computing workload_score / recovery_score / social_score:")

print(activity_clean[["student_id", "timestamp", "workload_score", "recovery_score", "social_score"]].head())



# Keep relevant columns and drop rows where ALL scores are missing

activity_clean = activity_clean[["student_id", "timestamp", "workload_score", "recovery_score", "social_score"]]
activity_clean = activity_clean.dropna( subset=["workload_score", "recovery_score", "social_score"], how="all" ).copy()



print("\nFinal cleaned Activity dataset (3 scores):")
print(activity_clean.head())




After computing workload_score / recovery_score / social_score:
  student_id           timestamp  workload_score  recovery_score  social_score
0        u56 2013-04-08 03:54:58             NaN             NaN           1.0
1        u56 2013-04-08 04:43:59             2.0             5.0           NaN
2        u56 2013-04-09 08:42:44             2.0             5.0           NaN
3        u56 2013-04-10 18:48:08             5.0             3.0           NaN
4        u56 2013-04-11 06:04:37             5.0             3.0           NaN

Final cleaned Activity dataset (3 scores):
  student_id           timestamp  workload_score  recovery_score  social_score
0        u56 2013-04-08 03:54:58             NaN             NaN           1.0
1        u56 2013-04-08 04:43:59             2.0             5.0           NaN
2        u56 2013-04-09 08:42:44             2.0             5.0           NaN
3        u56 2013-04-10 18:48:08             5.0             3.0           NaN
4        u56 2013-04-1

In [12]:
# Sleep dataset processing

# Print dataset name
print("\ DATASET: SLEEP")

# Describe dataset purpose
print("Shows students' self-reported sleep duration in hours")

# Start cleaning process
print("\nCleaning StudentLife Sleep...")

# Create a copy of raw dataset
sleep_clean = sleep_raw.copy()

# Remove irrelevant null column if present
if 'null' in sleep_clean.columns:
    sleep_clean = sleep_clean.drop(columns=['null'])

# Display initial dataset
print("\nInitial Sleep Dataset:")
print(sleep_clean)

# Convert response time to timestamp
sleep_clean['timestamp'] = pd.to_datetime(sleep_clean['resp_time'], unit='s')

# Show dataset after timestamp conversion
print("\nAfter converting resp_time to timestamp:")
print(sleep_clean)

\ DATASET: SLEEP
Shows students' self-reported sleep duration in hours

Cleaning StudentLife Sleep...

Initial Sleep Dataset:
       resp_time student_id hour                  location rate social
0     1364114760  Sleep_u00  NaN                       NaN  NaN    NaN
1     1364114765  Sleep_u00  NaN                       NaN  NaN    NaN
2     1364114453  Sleep_u00  NaN                       NaN  NaN    NaN
3     1364114775  Sleep_u00  NaN                       NaN  NaN    NaN
4     1364114365  Sleep_u00  NaN                       NaN  NaN    NaN
...          ...        ...  ...                       ...  ...    ...
1639  1367515556  Sleep_u35    6  43.70537226,-72.28333742    2      1
1640  1367604197  Sleep_u35    6  43.70537226,-72.28333742    2      1
1641  1368212337  Sleep_u35    8  43.65861498,-72.24869697    1      1
1642  1367867051  Sleep_u35    6  43.70517353,-72.28354394    1      1
1643  1369675857  Sleep_u35    4  43.70545577,-72.28364354    3      2

[1644 rows x 6 column

In [13]:
# Clean student_id by removing prefix
sleep_clean['student_id'] = sleep_clean['student_id'].str.replace('Sleep_', '')

# Show dataset after cleaning student_id
print("\nAfter cleaning student_id:")
print(sleep_clean)

# Convert hour column to numeric sleep_hours
sleep_clean['sleep_hours'] = pd.to_numeric(sleep_clean['hour'], errors='coerce')

# Show dataset after converting sleep hours
print("\nAfter converting hour to sleep_hours:")
print(sleep_clean)

# Recover missing sleep hours from null column if available
if 'null' in sleep_clean.columns:
    null_as_num = pd.to_numeric(sleep_clean['null'], errors='coerce')
    fill_mask = sleep_clean['sleep_hours'].isna() & null_as_num.notna()
    if fill_mask.any():
        sleep_clean.loc[fill_mask, 'sleep_hours'] = null_as_num.loc[fill_mask]

# Show dataset after recovery step
print("\nAfter recovering sleep_hours from null if applicable:")
print(sleep_clean)

# Keep only relevant columns and remove missing values
sleep_clean = sleep_clean[['student_id', 'timestamp', 'sleep_hours']].dropna()

# Display final cleaned dataset
print("\nFinal cleaned Sleep dataset:")
print(sleep_clean)


After cleaning student_id:
       resp_time student_id hour                  location rate social  \
0     1364114760        u00  NaN                       NaN  NaN    NaN   
1     1364114765        u00  NaN                       NaN  NaN    NaN   
2     1364114453        u00  NaN                       NaN  NaN    NaN   
3     1364114775        u00  NaN                       NaN  NaN    NaN   
4     1364114365        u00  NaN                       NaN  NaN    NaN   
...          ...        ...  ...                       ...  ...    ...   
1639  1367515556        u35    6  43.70537226,-72.28333742    2      1   
1640  1367604197        u35    6  43.70537226,-72.28333742    2      1   
1641  1368212337        u35    8  43.65861498,-72.24869697    1      1   
1642  1367867051        u35    6  43.70517353,-72.28354394    1      1   
1643  1369675857        u35    4  43.70545577,-72.28364354    3      2   

               timestamp  
0    2013-03-24 08:46:00  
1    2013-03-24 08:46:05  
2 

In [16]:
# Sahitya_week2 added code for feature engineering for stress dataset
print("CREATING WEEKLY STUDENTLIFE FEATURES (STRESS + SLEEP + ACTIVITY)")

for df, time_col in [  # loop through each dataframe and its timestamp column name

    (stress_clean, "timestamp"),  # pair: stress dataframe + timestamp column

    (activity_clean, "timestamp"),  # pair: activity dataframe + timestamp column

    (sleep_clean, "timestamp"),  # pair: sleep dataframe + timestamp column

]:

    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")  # convert to datetime;

stress_clean["stress_level"] = pd.to_numeric(stress_clean["stress_level"], errors="coerce")  # convert stress to numeric;

for col in ["workload_score","recovery_score","social_score"]:  # iterate through activity score columns

    if col in activity_clean.columns:  # check if the column exists in activity data

        activity_clean[col] = pd.to_numeric(activity_clean[col], errors="coerce")  # convert that column to numeric;

sleep_clean["sleep_hours"] = pd.to_numeric(sleep_clean["sleep_hours"], errors="coerce")  # convert sleep hours to numeric;

sleep_clean.head()

CREATING WEEKLY STUDENTLIFE FEATURES (STRESS + SLEEP + ACTIVITY)


,student_id,timestamp,sleep_hours
8,u00,2013-03-25 18:54:36,8.0
9,u00,2013-04-03 17:27:27,5.0
10,u00,2013-03-26 21:46:45,8.0
11,u00,2013-03-26 18:07:14,7.0
12,u00,2013-03-27 04:09:49,8.0
